In [1]:
import fireducks.pandas as pd
import numpy as np
import os

In [2]:
for day in [1, 2, 3, 4]:
    sc1 = pd.read_csv(f'./data/sc1_day{day}_us_og_v8.csv', index_col=0)
    state_df = pd.DataFrame(columns=[
        'state', 'perc_fuel_saved', 'num_trips', 
        'avg_fuel_saved_per_mile_smoothed', 'total_fuel_saved', 
        'total_cost_saved', 'avg_fuel_saved_per_100_miles'
    ])

    for state in sc1['state'].unique():
        state_rows = sc1.loc[sc1['state'] == state]
        state_row = {
            'state': state,   # ✅ use 'state' instead of 'index'
            'perc_fuel_saved': state_rows['perc_fuel_saved'].mean(),
            'num_trips': state_rows['num_trips'].sum(),
            'avg_fuel_saved_per_mile_smoothed': state_rows['avg_fuel_saved_per_mile_smoothed'].mean(),
            'total_fuel_saved': state_rows['total_fuel_saved'].sum(),
            'total_cost_saved': state_rows['total_cost_saved'].sum(),
            'avg_fuel_saved_per_100_miles': state_rows['avg_fuel_saved_per_100_miles'].mean()
        }

        # ✅ wrap in list to make a DataFrame with 1 row
        state_df = pd.concat([state_df, pd.DataFrame([state_row])], ignore_index=True)

    state_df.to_csv(f'./data/state_wise_day{day}_og.csv', index=False)

In [9]:
day3 = pd.read_csv('./data/sc1_day3_us_og_v8.csv', index_col=0)
day3['total_fuel_saved'].sum()

12003692.605730709

In [35]:
for DAY in [1,2,3,4]:

    day_states = pd.read_csv(f'./data/state_wise_day{DAY}_og.csv', index_col=0)
    day_states['total_fuel_saved'] = day_states['total_fuel_saved']/1000000
    day_states = day_states.drop(columns=['avg_fuel_saved_per_mile_smoothed'])
    day_states['total_cost_saved'] = day_states['total_cost_saved']/1000000
    day_states.index = day_states.index.str.title()
    day_states = day_states.rename(columns={'perc_fuel_saved': '% Fuel Saved',
                                             'num_trips': "Number of Trips",
                                             "avg_fuel_saved_per_100_miles": "Avg. Fuel Saved per 100 VMT",
                                             "total_fuel_saved": "Total Fuel Saved (million gallons)",
                                             "total_cost_saved": "Total Cost Saved (million USD)",
                                           "state": "State"})

    day_states = day_states.sort_values(by='Avg. Fuel Saved per 100 VMT', ascending=False)

    

    latex_code = day_states.to_latex(
        index=True,      # include the state names as index
        caption="State-level data summary",
        label="tab:state_data"
    )

    with open(f"./../cost-of-speed/data/day{DAY}_statewise.tex", "w") as f:
        f.write(latex_code)

In [31]:
df = pd.read_csv("./../cost-of-speed/data/statewise_summary.tex", index_col=0)
df.index = df.index.str.title()
latex_code = df.to_latex(
    index=True,
    caption="Average state-level results across 4 days",
    label="tab:state_data_summary"
)

with open("./../cost-of-speed/data/statewise_summary.tex", "w") as f:
    f.write(latex_code)

In [34]:
import pandas as pd

all_days = []

for DAY in [1, 2, 3, 4]:
    day_states = pd.read_csv(f'./data/state_wise_day{DAY}_og.csv', index_col=0)
    
    # scale + cleanup
    day_states['total_fuel_saved'] = day_states['total_fuel_saved']/1_000_000
    day_states['total_cost_saved'] = day_states['total_cost_saved']/1_000_000
    day_states = day_states.drop(columns=['avg_fuel_saved_per_mile_smoothed'])
    
    # rename to consistent column names
    day_states = day_states.rename(columns={
        'perc_fuel_saved': '% Fuel Saved',
        'num_trips': "Number of Trips",
        "avg_fuel_saved_per_100_miles": "Avg. Fuel Saved per 100 VMT",
        "total_fuel_saved": "Total Fuel Saved (million gallons)",
        "total_cost_saved": "Total Cost Saved (million USD)"
    })
    
    all_days.append(day_states)

# concatenate along a new axis
combined = pd.concat(all_days, axis=0, keys=[1, 2, 3, 4])

# aggregation rules
agg_dict = {col: 'mean' for col in combined.columns}
agg_dict["Number of Trips"] = 'sum'

# group by state index and apply rules
summary = combined.groupby(level=1).agg(agg_dict)
summary.index = summary.index.str.title()

# sort by fuel saved
summary = summary.sort_values(by="Avg. Fuel Saved per 100 VMT", ascending=False)

# export to LaTeX
latex_code = summary.to_latex(
    index=True,
    caption="Average state-level results across 4 days",
    label="tab:state_data_summary"
)

with open("./../cost-of-speed/data/statewise_summary.tex", "w") as f:
    f.write(latex_code)


In [4]:
state_df.sort_values(by='total_fuel_saved', ascending=False)

,state,perc_fuel_saved,num_trips,avg_fuel_saved_per_mile_smoothed,total_fuel_saved,total_cost_saved,avg_fuel_saved_per_100_miles
4,california,3.199290,2240458,0.001329,1.290633e+06,4.797283e+06,0.130728
41,texas,2.233859,3019525,0.000931,1.090985e+06,2.979481e+06,0.091751
8,florida,2.826511,1931820,0.001138,7.654803e+05,2.248981e+06,0.112037
31,new york,2.683742,1368148,0.001010,4.609825e+05,1.550745e+06,0.100928
12,illinois,2.769678,1195434,0.001028,4.101586e+05,1.216941e+06,0.099909
34,ohio,2.888733,1424424,0.001043,4.061111e+05,1.204932e+06,0.104042
37,pennsylvania,2.907711,1474302,0.001087,3.965817e+05,1.334101e+06,0.108363
32,north carolina,2.330623,764945,0.000899,3.944026e+05,1.158755e+06,0.089232
21,michigan,2.136111,1790006,0.000834,3.683796e+05,1.092982e+06,0.079058
20,massachusetts,3.981633,214132,0.001668,3.500094e+05,1.085029e+06,0.151851


In [4]:
sc1

,perc_fuel_saved,fuel_saving_per_day,fuel_use_frac,num_trips,avg_fuel_saved_per_mile,avg_fuel_saved_per_mile_smoothed,state,total_fuel_saved,total_cost_saved,usd_per_gallon,gallons_per_day,avg_fuel_saved_per_100_miles
djd,2.842197,185.111873,0.056529,25887,0.001076,0.001082,alabama,14573.277569,39406.142546,2.704,9.070503e+06,0.107643
dj3,3.099973,874.427318,0.201296,92347,0.001247,0.001252,alabama,56601.077138,153049.312580,2.704,9.070503e+06,0.124676
dj9,2.180226,68.617236,0.039733,16009,0.000838,0.000866,alabama,7857.571252,21246.872666,2.704,9.070503e+06,0.083752
dje,2.268784,136.302431,0.052145,23669,0.000892,0.000918,alabama,10730.828990,29016.161588,2.704,9.070503e+06,0.089222
djf,3.527525,1439.901282,0.289497,113078,0.001412,0.001401,alabama,92628.616896,250467.780088,2.704,9.070503e+06,0.141247
...,...,...,...,...,...,...,...,...,...,...,...,...
9xv,2.063412,11.683475,0.060082,1880,0.000800,0.000773,wyoming,1232.717350,3442.979560,2.793,9.943334e+05,0.079972
9xf,1.399121,4.029062,0.046776,2334,0.000553,0.000554,wyoming,650.751019,1817.547595,2.793,9.943334e+05,0.055270
9xg,1.393577,5.629838,0.059896,3912,0.000584,0.000583,wyoming,829.970524,2318.107674,2.793,9.943334e+05,0.058439
9xt,2.903328,6.754150,0.030988,1352,0.001269,0.001186,wyoming,894.571443,2498.538040,2.793,9.943334e+05,0.126883
